# 2. Zero-shot baselines

Stock whisper-small and whisper-large-v3, with no fine-tuning, on every speaker saying unseen sentences, plus 200 clips of everyday read English (LibriSpeech). About 30-45 minutes on an L4.

In [ ]:
# Setup: run this cell first. On Colab it clones the repo, installs packages and mounts Google Drive.
REPO_URL = "https://github.com/armaan-khoja/clear-voice"
SMOKE = False  # True = a quick check on a handful of clips (minutes, not hours)

import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/clear-voice"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/clear-voice"], check=True)
    os.chdir("/content/clear-voice")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/clearvoice"
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: downloads still work, uploading models will not.")
else:
    WORK = os.path.abspath("work")
sys.path.insert(0, os.getcwd())
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/models", exist_ok=True)
SPLITS = "splits"
LIMIT = 6 if SMOKE else None
print("Results and models go to", WORK)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "No GPU: Runtime > Change runtime type > L4 GPU, then rerun the setup cell.")

In [ ]:
from clearvoice.pipeline import load_torgo
parquets, meta = load_torgo(WORK)  # downloads TORGO (about 1.6 GB) the first time in each session
print(len(meta), "clips from", meta.speaker.nunique(), "speakers")

In [ ]:
import pandas as pd, torch
from clearvoice.asr import load_model
from clearvoice.pipeline import librispeech_check, predict_fold

general = []
for name in ["openai/whisper-small", "openai/whisper-large-v3"]:
    short = name.split("/")[1]
    model, processor = load_model(name)
    preds = pd.concat([predict_fold(model, processor, parquets, meta, SPLITS, fold, ["test_strict"], short,
                                    batch_size=32, limit=LIMIT) for fold in ["A", "B"]], ignore_index=True)
    preds.to_csv(f"{WORK}/results/preds_zeroshot_{short}.csv", index=False)
    ls = librispeech_check(model, processor, n=20 if SMOKE else 200, batch_size=32)
    general.append({"system": short, "librispeech_wer": ls["wer"], "n": ls["n"]})
    print(short, "done")
    del model
    torch.cuda.empty_cache()
pd.DataFrame(general).to_csv(f"{WORK}/results/librispeech_zeroshot.csv", index=False)
pd.DataFrame(general)

**First look:** dysarthric speakers, unseen sentences. Sentences scored by word error rate, single words by character error rate.

In [ ]:
from clearvoice.report import headline
headline({s: pd.read_csv(f"{WORK}/results/preds_zeroshot_{s}.csv") for s in ["whisper-small", "whisper-large-v3"]})